# Chapter 10: Option Pricing — Black-Scholes Model

This notebook implements closed-form European option pricing using the Black-Scholes formula.

In [ ]:
import sys
sys.path.append('../')

import numpy as np
import matplotlib.pyplot as plt

from src.chapter2_returns import simple_return, volatility
from src.chapter10_options import black_scholes_call, black_scholes_put
from src.data_utils import download_and_save_prices

## Black-Scholes Formula

For a European **call** option:

$$C = S \cdot N(d_1) - K e^{-rT} \cdot N(d_2)$$

For a European **put** option:

$$P = K e^{-rT} \cdot N(-d_2) - S \cdot N(-d_1)$$

Where:

$$d_1 = \frac{\ln(S/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$

- $S$: current stock price
- $K$: strike price
- $T$: time to expiration (years)
- $r$: risk-free rate
- $\sigma$: annualized volatility
- $N(\cdot)$: standard normal CDF

## Estimate Volatility from Market Data

We estimate $\sigma$ from historical UBS returns.

In [ ]:
prices, _ = download_and_save_prices('UBS', '2022-01-01', '2024-01-01', data_dir='../data')
returns = simple_return(prices)

sigma = volatility(returns) * np.sqrt(252)   # annualized historical volatility
S = prices[-1]                                # most recent price as current price

print(f"Current UBS price (S):      ${S:.2f}")
print(f"Annualized volatility (σ):  {sigma*100:.2f}%")

## Price a Single Option

In [ ]:
K = round(S)    # at-the-money strike
T = 0.5         # 6 months to expiration
r = 0.015       # risk-free rate (1.5%)

call_price = black_scholes_call(S, K, T, r, sigma)
put_price = black_scholes_put(S, K, T, r, sigma)

print(f"Option Parameters:")
print(f"  S (current price): ${S:.2f}")
print(f"  K (strike):        ${K:.2f}")
print(f"  T (years):         {T}")
print(f"  r (risk-free):     {r*100:.1f}%")
print(f"  σ (volatility):    {sigma*100:.2f}%")
print(f"\nBlack-Scholes Prices:")
print(f"  Call price: ${call_price:.4f}")
print(f"  Put price:  ${put_price:.4f}")
print(f"\nPut-Call Parity check:")
print(f"  C - P = {call_price - put_price:.4f}")
print(f"  S - K*e^(-rT) = {S - K*np.exp(-r*T):.4f}")

## Option Price vs Stock Price

How the call and put prices change as the underlying price varies.

In [ ]:
price_range = np.linspace(S * 0.5, S * 1.5, 200)

call_prices = [black_scholes_call(s, K, T, r, sigma) for s in price_range]
put_prices = [black_scholes_put(s, K, T, r, sigma) for s in price_range]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(price_range, call_prices, color='green', linewidth=2)
axes[0].axvline(x=K, color='gray', linestyle='--', linewidth=1, label=f'Strike K=${K}')
axes[0].axvline(x=S, color='red', linestyle=':', linewidth=1, label=f'Current S=${S:.2f}')
axes[0].set_title('Call Option Price vs Stock Price', fontweight='bold')
axes[0].set_xlabel('Stock Price ($)')
axes[0].set_ylabel('Call Price ($)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(price_range, put_prices, color='red', linewidth=2)
axes[1].axvline(x=K, color='gray', linestyle='--', linewidth=1, label=f'Strike K=${K}')
axes[1].axvline(x=S, color='blue', linestyle=':', linewidth=1, label=f'Current S=${S:.2f}')
axes[1].set_title('Put Option Price vs Stock Price', fontweight='bold')
axes[1].set_xlabel('Stock Price ($)')
axes[1].set_ylabel('Put Price ($)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Option Price vs Volatility (Vega)

Both calls and puts increase in value with higher volatility.

In [ ]:
vol_range = np.linspace(0.05, 0.80, 100)

call_vs_vol = [black_scholes_call(S, K, T, r, v) for v in vol_range]
put_vs_vol = [black_scholes_put(S, K, T, r, v) for v in vol_range]

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(vol_range * 100, call_vs_vol, color='green', linewidth=2, label='Call')
ax.plot(vol_range * 100, put_vs_vol, color='red', linewidth=2, label='Put')
ax.axvline(x=sigma * 100, color='gray', linestyle='--', linewidth=1, label=f'Historical σ = {sigma*100:.1f}%')

ax.set_xlabel('Volatility (%)')
ax.set_ylabel('Option Price ($)')
ax.set_title('Option Price vs Volatility', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:

1. **d1 and d2**: Intermediate terms encoding moneyness, time, and volatility
2. **Call pricing**: $C = S N(d_1) - K e^{-rT} N(d_2)$
3. **Put pricing**: $P = K e^{-rT} N(-d_2) - S N(-d_1)$
4. **Put-Call Parity**: $C - P = S - K e^{-rT}$ — verified numerically
5. **Vega**: Both call and put prices increase with volatility